# 프롬프트 템플릿용 합성 테스트 데이터 만들기

대략 이런 형태의 프롬프트가 있다고 해 봅시다.

"""Here's some things I want you to analyze:

<thing>
{{thing1}}
</thing>
<thing>
{{thing2}}
</thing>

These things are [description of things]. Please read them carefully and [do some task]."""

여기서 thing1과 thing2를 "변수"라고 부르겠습니다. 여러분은 thing1과 thing2에 어떤 값이 들어와도 프롬프트가 잘 동작하기를 바랄 것입니다.

이 프롬프트 템플릿을 어떻게 테스트할 수 있을까요? 대입해 볼 실제 값이 있을 수도 있습니다. 하지만 없을 수도 있고, 있더라도 개인정보 문제로 그 값으로 테스트할 수 없을 수도 있습니다. 걱정하지 마세요. Claude가 만들어 줄 수 있습니다! 이 쿡북에서는 Claude와 Claude API로 프롬프트용 합성 테스트 데이터를 생성하는 방법을 다룹니다. 템플릿에서 변수를 추출하고, 예시 블록을 구성하고, 테스트 케이스를 생성하고, 결과를 반복적으로 다듬는 함수들이 포함되어 있습니다. 이렇게 하면 두 가지 이점이 있습니다.

1. 프롬프트 평가
현실적인 예시에서 Claude가 어떻게 동작하는지 이 테스트 케이스로 확인할 수 있습니다.

2. 멀티샷 예시를 통한 프롬프트 개선
Claude에 예시를 제공하는 것은 성능을 높이는 가장 좋은 방법일 것입니다. 이 노트북은 현실적인 입력을 만들어 주는데, 이상적인 입력/출력 쌍을 얻는 일의 절반은 여기에 달려 있습니다.

In [ ]:
% pip install anthropic IPython

In [1]:
import re

import anthropic

# Enter your API key here
api_key = ""
CLIENT = anthropic.Anthropic(api_key=api_key)
MODEL_NAME = "claude-sonnet-4-6"

먼저 이 노트북 전반에서 사용할 헬퍼 함수들을 정의하겠습니다.

In [2]:
# First, we have the `extract_variables` function,
# It takes in a prompt template and extracts the double-mustache-bracketed "variables" contained.
def extract_variables(prompt_template):
    """Extract variables from a prompt template."""
    pattern = r"{{([^}]+)}}"
    variables = re.findall(pattern, prompt_template)
    return set(variables)


# Next, we have `construct_variables_names`, which just joins them together connected by newlines.
def construct_variables_names(prompt_template):
    """Construct a string of variable names from a prompt template."""
    variables = extract_variables(prompt_template)
    return "\n".join(variables)


# The `construct_variables_block` function takes in the list of variables, and constructs a "variables block"
# The variables block might look like this, if the variables were 'animal' and 'topic':
"""
<animal>
[a full, complete, value for the variable "animal"]
</animal>
<topic>
[a full, complete, value for the variable "topic"]
</topic>
"""


def construct_variables_block(prompt_template):
    """Construct a variables block for the synthetic test data prompt."""
    variables = extract_variables(prompt_template)
    output = ""
    for v in variables:
        output += f"<{v}>\n"
        output += f'[a full, complete, value for the variable "{v}". (You do not need to repeat the variable name inside the tags.)]\n'
        output += f"</{v}>\n"
    return output.strip()


# `construct_examples` takes a dictionary of {variable: value} and constructs an XML-formatted example.
# E.g. if the dict is
# {'animal': 'cat', 'topic': 'movement patterns'}, then the example would be
"""
<example>
<variables>
<animal>
cat
</animal>
<topic>
movement patterns
</topic>
</variables>
</example>
"""


def construct_example_block(variable_dict):
    """Construct an example block from a dictionary of variables."""
    output = "<example>\n<variables>\n"
    for k, v in variable_dict.items():
        output += f"<{k}>\n{v}\n</{k}>\n"
    output = output.strip()
    output += "\n</variables>\n</example>"
    return output

## 데이터 생성용 프롬프트 템플릿

이 프롬프트 템플릿들의 기본 발상은, 변수가 들어 있는 사용자 제출 프롬프트 템플릿을 받아 그 템플릿을 채울 변숫값을 만들어 내는 것입니다.

아래에는 실제로 두 개의 프롬프트 템플릿이 있습니다. 하나는 사용자가 예시 변숫값을 이미 제공한 경우를 가정한 것이고, 다른 하나는 그렇지 않은 경우를 위한 것입니다.

두 템플릿의 공통점은, 먼저 Claude에 상황에 대한 맥락을 제공한 뒤, 테스트 케이스를 출력하기 전에 각 변수의 명세를 하나씩, 그리고 사용자가 제공한 프롬프트 템플릿 전체를 꼼꼼히 검토하도록 지시한다는 점입니다.

In [3]:
# Formatting Prompt Templates for Synthetic Evaluations

# This function prepares the prompt template for generating synthetic test data.


def format_prompt_template_for_synth_evals(prompt_template, examples=None):
    """Format a prompt template for synthetic evaluations."""
    synth_test_data_prompt_template_with_example = """<Prompt Template>
{{PROMPT_TEMPLATE}}
</Prompt Template>

Your job is to construct a test case for the prompt template above. This template contains "variables", which are placeholders to be filled in later. In this case, the variables are:

<variables>
{{CONSTRUCT_VARIABLES_NAMES}}
</variables>

Here are the example test cases provided by the user.
<examples>
{{EXAMPLES}}
</examples>

First, in <planning> tags, do the following:

1. Summarize the prompt template. What is the goal of the user who created it?
2. For each variable in <variables>, carefully consider what a paradigmatic, realistic example of that variable would look like. You'll want to note who will be responsible "in prod" for supplying values. Written by a human "end user"? Downloaded from a website? Extracted from a database? Think about things like length, format, and tone in addition to semantic content. Use the examples provided by the user to guide this exercise. The goal is to acquire a sense of the statistical distribution the examples are being drawn from. The example you write should be drawn from that same distribution, but sufficiently different from the examples that it provides additional signal. A tricky balancing act, but I have faith in you.

Once you're done, output a test case for this prompt template with a full, complete, value for each variable. The output format should consist of a tagged block for each variable, with the value inside the block, like the below:

<variables>
{{CONSTRUCT_VARIABLES_BLOCK}}
</variables>"""

    synth_test_data_prompt_template_without_example = """<Prompt Template>
{{PROMPT_TEMPLATE}}
</Prompt Template>

Your job is to construct a test case for the prompt template above. This template contains "variables", which are placeholders to be filled in later. In this case, the variables are:

<variables>
{{CONSTRUCT_VARIABLES_NAMES}}
</variables>

First, in <planning> tags, do the following:

1. Summarize the prompt template. What is the goal of the user who created it?
2. For each variable in <variables>, carefully consider what a paradigmatic, realistic example of that variable would look like. You'll want to note who will be responsible "in prod" for supplying values. Written by a human "end user"? Downloaded from a website? Extracted from a database? Think about things like length, format, and tone in addition to semantic content.

Then, output a test case for this prompt template with a full, complete, value for each variable. The output format should consist of a tagged block for each variable, with the value inside the block, like the below:
<variables>
{{CONSTRUCT_VARIABLES_BLOCK}}
</variables>"""

    if examples:
        examples_block = "\n".join([construct_example_block(example) for example in examples])
        return (
            synth_test_data_prompt_template_with_example.replace(
                "{{PROMPT_TEMPLATE}}", prompt_template
            )
            .replace("{{CONSTRUCT_VARIABLES_NAMES}}", construct_variables_names(prompt_template))
            .replace("{{CONSTRUCT_VARIABLES_BLOCK}}", construct_variables_block(prompt_template))
            .replace("{{EXAMPLES}}", examples_block)
        )
    else:
        return (
            synth_test_data_prompt_template_without_example.replace(
                "{{PROMPT_TEMPLATE}}", prompt_template
            )
            .replace("{{CONSTRUCT_VARIABLES_NAMES}}", construct_variables_names(prompt_template))
            .replace("{{CONSTRUCT_VARIABLES_BLOCK}}", construct_variables_block(prompt_template))
        )

다음으로, 알맞은 프롬프트 템플릿을 채워 Claude를 호출하는 간단한 헬퍼 함수를 하나 더 만듭니다.

In [4]:
def get_test_data(prompt_template, examples, custom_planning=None):
    """Generate test data using the Claude API."""
    synth_eval_prompt_ready = format_prompt_template_for_synth_evals(prompt_template, examples)

    messages = [
        {
            "role": "user",
            "content": synth_eval_prompt_ready,
        }
    ]
    if custom_planning:
        messages.append(
            {
                "role": "assistant",
                "content": custom_planning,
            }
        )

    message = (
        CLIENT.messages.create(
            max_tokens=4000,
            messages=messages,
            model=MODEL_NAME,
            temperature=1,
        )
        .content[0]
        .text
    )

    return message

In [5]:
# We'll use this function to sample Claude's response to the filled-in template,
# once we have our example values/test case.


def call_claude_with_template(prompt_template, variables):
    """Call Claude with a filled prompt template."""
    filled_template = prompt_template
    for var, value in variables.items():
        filled_template = filled_template.replace(f"{{{{{var}}}}}", value)

    message = (
        CLIENT.messages.create(
            max_tokens=4000,
            messages=[
                {
                    "role": "user",
                    "content": filled_template,
                }
            ],
            model=MODEL_NAME,
            temperature=0.7,
        )
        .content[0]
        .text
    )

    return message

이제 조각들을 하나로 맞춰 보겠습니다. 우선 여기에 프롬프트 템플릿을 입력하세요.

In [6]:
# Replace this with your prompt template!
# Use double-brackets to indicate variables
# Here's an example:
prompt_template = """You are a customer support bot for Acme Corporation.
Here is an FAQ with Acme's relevant policies:

<documents>
{{DOCUMENTS}}
</documents>

Please respond to this customer support question using details from the policies:

<question>
{{QUESTION}}
</question>"""

variables = extract_variables(prompt_template)
print("\nIdentified variables:")
for var in variables:
    print(f"- {var}")


Identified variables:
- DOCUMENTS
- QUESTION


다음으로, 입력과 이상적인 출력으로 이뤄진 "골든 예시"가 있다면 여기에 입력하면 됩니다. 지금은 해당 코드가 주석 처리되어 있습니다.

In [7]:
planning_text = None
USER_EXAMPLES = []

# if input("\nDo you want to provide an example value for your variables? (y/n): ").lower() == 'y':
#     example = {}
#     for var in variables:
#         example[var] = input(f"Enter an example value for {var}: ")
#     USER_EXAMPLES.append(example)

이제 이 정보로 테스트 케이스 생성 프롬프트 템플릿을 채워 테스트 케이스를 받아 봅니다!

In [8]:
result = get_test_data(prompt_template, USER_EXAMPLES, planning_text)

이제 테스트 케이스와, Claude가 그것을 만들기 위해 세운 계획을 함께 살펴보겠습니다.

In [10]:
planning_match = re.search(r"<planning>(.*?)</planning>", result, re.DOTALL)
if planning_match and not planning_text:
    planning_text = "<planning>\n" + planning_match.group(1).strip() + "\n</planning>"

extracted_variables = {}
for var in variables:
    var_match = re.search(f"<{var}>(.*?)</{var}>", result[result.index("<variables>") :], re.DOTALL)
    if var_match:
        extracted_variables[var] = var_match.group(1).strip()

USER_EXAMPLES.append(extracted_variables)

print("~~~~~~~~~~~\nGenerated test case:\n~~~~~~~~~~~")
for var, value in extracted_variables.items():
    print(f"{var}:\n{value}\n")

print("~~~~~~~~~~~\nPlanning:\n~~~~~~~~~~~")
print(planning_text)

~~~~~~~~~~~
Generated test case:
~~~~~~~~~~~
DOCUMENTS:
Return Policy
- Items may be returned within 30 days of purchase with original receipt
- Items must be unused and in original packaging
- Shipping costs are non-refundable
- Gift cards are non-returnable

Shipping Information
- Standard shipping (5-7 business days): Free on orders over $50
- Express shipping (2-3 business days): $12.99
- Overnight shipping (next business day): $24.99
- We ship to continental US only
- Alaska and Hawaii orders incur additional $15 fee

Payment Methods
- We accept Visa, Mastercard, American Express, and PayPal
- Payment is processed at time of order
- Gift cards cannot be used for partial payment

QUESTION:
Hi, I ordered a sweater last week but it doesn't fit right. Can I return it? And will I get refunded for the shipping I paid? Thanks!

~~~~~~~~~~~
Planning:
~~~~~~~~~~~
<planning>
1. Prompt Template Summary:
This template creates a customer service chatbot for Acme Corporation that answers custom

여기서부터는 몇 가지 방향으로 나아갈 수 있습니다. 테스트 케이스를 더 생성할 수도 있고, Claude의 계획 로직을 수정할 수도 있습니다. 여기서는 계획 로직을 조금 손봐 보겠습니다. 예를 들어 ACME의 문서가 번호 매긴 줄을 사용한다는 사실을 알고 있다고 해 봅시다. 현실적으로 시도해 볼 만한 다른 변경으로는 다음과 같은 것들이 있습니다.

- 문서를 더 길고 상세하게 작성하라고 Claude가 스스로에게 지시하게 하기
- 고객 지원 문의의 격식 수준을 더 높이거나 낮추라고 Claude가 스스로에게 지시하게 하기

In [11]:
planning_text = planning_text.replace(
    "each with a question and answer format",
    "each with a question and answer format and associated number.",
)
# You might have slightly different planning text and therefore need to rewrite the replace.

예시를 초기화하되, 이 계획 텍스트를 프리필로 사용하겠습니다. (샘플링 시간을 조금 아낄 수 있습니다.)

In [12]:
USER_EXAMPLES = []
result = get_test_data(prompt_template, USER_EXAMPLES, planning_text)

이제 새 결과를 확인해 보겠습니다.

In [13]:
# Copied and pasted from a cell above.
planning_match = re.search(r"<planning>(.*?)</planning>", result, re.DOTALL)
if planning_match and not planning_text:
    planning_text = "<planning>\n" + planning_match.group(1).strip() + "\n</planning>"

extracted_variables = {}
for var in variables:
    var_match = re.search(f"<{var}>(.*?)</{var}>", result[result.index("<variables>") :], re.DOTALL)
    if var_match:
        extracted_variables[var] = var_match.group(1).strip()

USER_EXAMPLES.append(extracted_variables)

print("~~~~~~~~~~~\nGenerated test case:\n~~~~~~~~~~~")
for var, value in extracted_variables.items():
    print(f"{var}:\n{value}\n")

print("~~~~~~~~~~~\nPlanning:\n~~~~~~~~~~~")
print(planning_text)

~~~~~~~~~~~
Generated test case:
~~~~~~~~~~~
DOCUMENTS:
Return Policy
- Items may be returned within 30 days of purchase with original receipt
- Items must be unused and in original packaging
- Shipping costs are non-refundable
- Store credit will be issued for items returned without receipt

Shipping Information
- Standard shipping (5-7 business days): $5.99
- Express shipping (2-3 business days): $12.99
- Free standard shipping on orders over $50
- We currently ship only within the continental United States
- Alaska and Hawaii orders subject to additional fees

Payment Methods
- We accept Visa, Mastercard, American Express, and PayPal
- Gift cards cannot be used for online purchases
- Payment is processed at time of order
- All prices are in USD

QUESTION:
Hi, I ordered a sweater last week but it doesn't fit right. Can I return it? I still have the tags on it but I threw away the receipt. Thanks!

~~~~~~~~~~~
Planning:
~~~~~~~~~~~
<planning>
1. Prompt Template Summary:
This template 

좋습니다. 번호를 매긴 Q&A 형태로 만들어 냈습니다!

예시를 하나 더 만들어 보겠습니다. 이번에는 이미 갖고 있는 예시를 활용하므로, 흥미롭게 다른 결과가 나오기를 기대해 봅니다.

In [14]:
result = get_test_data(prompt_template, USER_EXAMPLES, planning_text)

In [15]:
# Copied and pasted from a cell above.
planning_match = re.search(r"<planning>(.*?)</planning>", result, re.DOTALL)
if planning_match and not planning_text:
    planning_text = "<planning>\n" + planning_match.group(1).strip() + "\n</planning>"

extracted_variables = {}
for var in variables:
    var_match = re.search(f"<{var}>(.*?)</{var}>", result[result.index("<variables>") :], re.DOTALL)
    if var_match:
        extracted_variables[var] = var_match.group(1).strip()

USER_EXAMPLES.append(extracted_variables)

print("~~~~~~~~~~~\nGenerated test case:\n~~~~~~~~~~~")
for var, value in extracted_variables.items():
    print(f"{var}:\n{value}\n")

print("~~~~~~~~~~~\nPlanning:\n~~~~~~~~~~~")
print(planning_text)

~~~~~~~~~~~
Generated test case:
~~~~~~~~~~~
DOCUMENTS:
Product Warranty
- All electronics come with a 1-year limited manufacturer warranty
- Warranty covers defects in materials and workmanship
- Warranty does not cover accidental damage or misuse
- Extended warranty available for purchase within 30 days

Price Match Policy
- We match prices from authorized retailers
- Item must be identical model/color/specification
- Must be in stock at competitor's store
- Online retailers excluded from price matching
- Price match requests must be made at time of purchase

Order Cancellation
- Orders can be cancelled within 2 hours of placement
- Once order is shipped, cancellation not possible
- Cancelled orders refunded to original payment method
- Processing time for refunds: 3-5 business days
- Contact customer service for cancellation requests

QUESTION:
Hello, I bought a laptop from your store 3 weeks ago and it keeps shutting down randomly. It's still under warranty, right? What do I need t

여전히 ACME사에 대한 내용이지만, 질문도 지식 베이스도 달라졌습니다.

여기서부터는 무엇이든 할 수 있습니다. 코드를 반복 실행해 테스트 케이스를 더 만들고, 계획을 더 다듬고, 이 테스트 케이스로 Claude를 평가하고, 만들어 낸 테스트 케이스를 골든 답변과 함께 멀티샷 예시로 프롬프트에 넣을 수 있습니다.

골든 답변은 처음부터 직접 작성해도 되고, Claude에게 답변을 쓰게 한 뒤 취향에 맞게 다듬어도 됩니다. 프롬프트 캐싱이 등장한 지금, 성능을 높이기 위해 프롬프트에 예시를 잔뜩 넣기에 이보다 좋은 때는 없었습니다.